In [ ]:
# 실행 시간 측정: 단일 AEDTPLT export
import time

start_aedtplt = time.perf_counter()

# 첫 번째 variation만 추출
test_table_1_aedtplt = ParametricTable.iloc[:1]

print("=" * 70)
print(f"⏱️  AEDTPLT Export 시간 측정 (1개 Variation)")
print("=" * 70)

# AEDTPLT export 실행
aedtplt_files_test = export_aedtplt_for_variations(
    m2d_obj=m2d,
    parametric_table=test_table_1_aedtplt,
    quantity="Mag_B",
    solution="Setup1 : Transient",
    assignment="AllObjects",
    output_dir=r"D:\KDHe10\e10_example\AEDTPLT_Exports_Test",
    intrinsics={"Time": "0.06s"}
)

elapsed_aedtplt = time.perf_counter() - start_aedtplt

print(f"\n{'='*70}")
print("📊 AEDTPLT 성능 측정 결과")
print(f"{'='*70}")
print(f"✅ 총 소요 시간: {elapsed_aedtplt:.3f} 초")
print(f"📦 생성된 파일: {len(aedtplt_files_test)}개")
print(f"\n💡 전체 {len(ParametricTable)}개 variation 예상 시간:")
print(f"   약 {elapsed_aedtplt * len(ParametricTable):.1f} 초 ({elapsed_aedtplt * len(ParametricTable) / 60:.1f} 분)")
print(f"{'='*70}")

# Maxwell 2D Field Data Export

## Setup: AEDT Connection & Utilities

1. Mesh는 case로 내보내기
2. fld로 field내보내기

## 📚 패키지 정보

이 노트북은 `aedt_utils` 패키지를 사용합니다.

패키지 위치: `d:\KangDH\Emlab_emach\pyAEDT\aedt_utils\`

### 주요 기능:
- **Connection**: AEDT Desktop 자동 연결
- **Maxwell**: Maxwell 2D 디자인 자동 연결

### 사용 가능한 함수:
```python
from aedt_utils import (
    smartAedtConnector,      # 스마트 연결 (권장)
    quickConnect,            # 빠른 연결
    getDesktopConnection,    # Desktop 연결
    checkCurrentDesktopStatus, # 상태 확인
    getRunningMaxwell2d,     # Maxwell 2D 연결
)
```


In [ ]:
import sys
sys.path.append(r'd:\KangDH\Emlab_emach\pyAEDT')

# AEDT Utils 패키지 import
from aedt_utils import (
    smartAedtConnector,
    quickConnect,
    getDesktopConnection,
    checkCurrentDesktopStatus,
    getAedtProcessesDetailed,
)

# 기본 설정
AEDT_VERSION = "2025.2"
NUM_CORES = 8
NG_MODE = False  # Open AEDT UI when it is launched.


## Load Maxwell 2D Model

In [ ]:
# e10 Model
AEDT_VERSION = "2025.2"
NUM_CORES = 8
NG_MODE = False  # Open AEDT UI when it is launched.

In [ ]:
# aedt_file=r"D:\KDHe10\e10_example\e10_tutorial_ANSYSEM_2D.aedt"
aedt_file=r"E:\KDH\e10\e10_DOE\e10_DOE.opd\AMOP\Design0001\e10_DOE.aedt"
m2d = ansys.aedt.core.Maxwell2d(
    project=aedt_file,
    version=AEDT_VERSION,
    new_desktop=False,
    non_graphical=NG_MODE
)

## Utility: 현재 실행중인 Maxwell2d 객체 자동 연결 함수

In [1]:
# Maxwell 2D 유틸리티 import
from aedt_utils import getRunningMaxwell2d

# 사용 예시
print("="*70)
print("🔌 현재 실행 중 Maxwell2d 객체 가져오기")
print("="*70)

m2d_running = getRunningMaxwell2d()

if m2d_running:
    print(f"Design Type: {m2d_running.design_type}")
    print(f"Variables: {list(m2d_running.variable_manager.variables.keys())[:5]} ...")
else:
    print("⚠️ Maxwell2d 객체를 가져오지 못했습니다.")

m2d=m2d_running


🔌 현재 실행 중 Maxwell2d 객체 가져오기
PyAEDT INFO: Python version 3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)].
PyAEDT INFO: PyAEDT version 0.22.0.
PyAEDT INFO: Initializing new Desktop session.
PyAEDT WARNING: Argument `specified_version` is deprecated for method `__init__`; use `version` instead.
PyAEDT WARNING: Argument `new_desktop_session` is deprecated for method `__init__`; use `new_desktop` instead.
PyAEDT INFO: Log on console is enabled.
PyAEDT INFO: Log on file C:\Users\user\AppData\Local\Temp\pyaedt_user_729b9761-b8d6-4f72-adf1-11416fb34a76.log is enabled.
PyAEDT INFO: Log on AEDT is disabled.
PyAEDT INFO: Debug logger is disabled. PyAEDT methods will not be logged.
PyAEDT INFO: PyAEDT version 0.22.0.
PyAEDT INFO: Initializing new Desktop session.
PyAEDT WARNING: Argument `specified_version` is deprecated for method `__init__`; use `version` instead.
PyAEDT WARNING: Argument `new_desktop_session` is deprecated for method `__init__`; use `new_desktop

## AEDT List 추출

E:\KDH\e10\e10_DOE\e10_DOE.opd\AMOP

In [3]:
# AEDT 파일 관리 유틸리티 import
import pandas as pd
from aedt_file_utils import find_aedt_files, find_lock_files, remove_lock_files, find

# AEDT 파일 검색 예시
target_dir = r"E:\KDH\e10\e10_DOE\e10_DOE.opd\AMOP"
aedt_files = find_aedt_files(target_dir, recursive=True)
lock_files = find_lock_files(target_dir)
result = remove_lock_files(lock_files)
# print(result['message'])
import ansys.aedt.core
from concurrent.futures import ProcessPoolExecutor, as_completed
import time

# 별도 파일에서 함수 import (ProcessPoolExecutor 호환)

from aedt_parallel_processor import process_aedt_file
# DataFrame으로 변환하여 표시
if aedt_files:
    df_aedt = pd.DataFrame(aedt_files)
# Lock 파일 찾기 및 삭제 예시
file_paths = df_aedt['full_path'].tolist()
file_paths

🔍 AEDT 파일 검색 중...
📂 경로: E:\KDH\e10\e10_DOE\e10_DOE.opd\AMOP
🔄 재귀 검색: 예

✅ 총 300개의 .aedt 파일 발견

📋 발견된 파일 목록:

  1. e10_DOE.aedt
     전체 경로: E:\KDH\e10\e10_DOE\e10_DOE.opd\AMOP\Design0001\e10_DOE.aedt
     크기: 1.71 MB
     수정: 2025-11-19 15:48:20

  2. e10_DOE.aedt
     전체 경로: E:\KDH\e10\e10_DOE\e10_DOE.opd\AMOP\Design0002\e10_DOE.aedt
     크기: 1.71 MB
     수정: 2025-11-19 15:48:54

  3. e10_DOE.aedt
     전체 경로: E:\KDH\e10\e10_DOE\e10_DOE.opd\AMOP\Design0003\e10_DOE.aedt
     크기: 1.70 MB
     수정: 2025-11-19 15:56:39

  4. e10_DOE.aedt
     전체 경로: E:\KDH\e10\e10_DOE\e10_DOE.opd\AMOP\Design0004\e10_DOE.aedt
     크기: 1.71 MB
     수정: 2025-11-17 10:46:15

  5. e10_DOE.aedt
     전체 경로: E:\KDH\e10\e10_DOE\e10_DOE.opd\AMOP\Design0005\e10_DOE.aedt
     크기: 1.73 MB
     수정: 2025-11-20 10:05:01

  6. e10_DOE.aedt
     전체 경로: E:\KDH\e10\e10_DOE\e10_DOE.opd\AMOP\Design0006\e10_DOE.aedt
     크기: 1.71 MB
     수정: 2025-11-17 10:49:38

  7. e10_DOE.aedt
     전체 경로: E:\KDH\e10\e10_DOE\e10_DOE.opd\AMOP\Des

['E:\\KDH\\e10\\e10_DOE\\e10_DOE.opd\\AMOP\\Design0001\\e10_DOE.aedt',
 'E:\\KDH\\e10\\e10_DOE\\e10_DOE.opd\\AMOP\\Design0002\\e10_DOE.aedt',
 'E:\\KDH\\e10\\e10_DOE\\e10_DOE.opd\\AMOP\\Design0003\\e10_DOE.aedt',
 'E:\\KDH\\e10\\e10_DOE\\e10_DOE.opd\\AMOP\\Design0004\\e10_DOE.aedt',
 'E:\\KDH\\e10\\e10_DOE\\e10_DOE.opd\\AMOP\\Design0005\\e10_DOE.aedt',
 'E:\\KDH\\e10\\e10_DOE\\e10_DOE.opd\\AMOP\\Design0006\\e10_DOE.aedt',
 'E:\\KDH\\e10\\e10_DOE\\e10_DOE.opd\\AMOP\\Design0007\\e10_DOE.aedt',
 'E:\\KDH\\e10\\e10_DOE\\e10_DOE.opd\\AMOP\\Design0008\\e10_DOE.aedt',
 'E:\\KDH\\e10\\e10_DOE\\e10_DOE.opd\\AMOP\\Design0009\\e10_DOE.aedt',
 'E:\\KDH\\e10\\e10_DOE\\e10_DOE.opd\\AMOP\\Design0010\\e10_DOE.aedt',
 'E:\\KDH\\e10\\e10_DOE\\e10_DOE.opd\\AMOP\\Design0011\\e10_DOE.aedt',
 'E:\\KDH\\e10\\e10_DOE\\e10_DOE.opd\\AMOP\\Design0012\\e10_DOE.aedt',
 'E:\\KDH\\e10\\e10_DOE\\e10_DOE.opd\\AMOP\\Design0013\\e10_DOE.aedt',
 'E:\\KDH\\e10\\e10_DOE\\e10_DOE.opd\\AMOP\\Design0014\\e10_DOE.aedt',
 'E:\\

##  AEDT 해석

여러 AEDT 파일을 병렬로 열고 Excitation 설정 및 Parametric Sweep을 자동으로 적용합니다.

##  실행

CSV 파일에서 전류와 각도 데이터를 읽어 Optimetrics 설정을 생성합니다.

- **전류**: 5 steps
- **각도**: 6 steps
- **총 Variations**: 30개 (5 × 6)

 전류와 각도 변수 추출 및 Sweep 범위 설정

 유틸리티 함수: 객체 속성 검색

`find()` 함수를 사용하여 객체의 속성이나 메서드를 쉽게 검색할 수 있습니다.

In [4]:
m2d.project_path

'E:/KDH/e10/e10_DOE/e10_DOE.opd/AMOP/Design0007/'

In [8]:
file_path = Path(1)


NameError: name 'Path' is not defined

In [6]:
file_paths

['E:\\KDH\\e10\\e10_DOE\\e10_DOE.opd\\AMOP\\Design0001\\e10_DOE.aedt',
 'E:\\KDH\\e10\\e10_DOE\\e10_DOE.opd\\AMOP\\Design0002\\e10_DOE.aedt',
 'E:\\KDH\\e10\\e10_DOE\\e10_DOE.opd\\AMOP\\Design0003\\e10_DOE.aedt',
 'E:\\KDH\\e10\\e10_DOE\\e10_DOE.opd\\AMOP\\Design0004\\e10_DOE.aedt',
 'E:\\KDH\\e10\\e10_DOE\\e10_DOE.opd\\AMOP\\Design0005\\e10_DOE.aedt',
 'E:\\KDH\\e10\\e10_DOE\\e10_DOE.opd\\AMOP\\Design0006\\e10_DOE.aedt',
 'E:\\KDH\\e10\\e10_DOE\\e10_DOE.opd\\AMOP\\Design0007\\e10_DOE.aedt',
 'E:\\KDH\\e10\\e10_DOE\\e10_DOE.opd\\AMOP\\Design0008\\e10_DOE.aedt',
 'E:\\KDH\\e10\\e10_DOE\\e10_DOE.opd\\AMOP\\Design0009\\e10_DOE.aedt',
 'E:\\KDH\\e10\\e10_DOE\\e10_DOE.opd\\AMOP\\Design0010\\e10_DOE.aedt',
 'E:\\KDH\\e10\\e10_DOE\\e10_DOE.opd\\AMOP\\Design0011\\e10_DOE.aedt',
 'E:\\KDH\\e10\\e10_DOE\\e10_DOE.opd\\AMOP\\Design0012\\e10_DOE.aedt',
 'E:\\KDH\\e10\\e10_DOE\\e10_DOE.opd\\AMOP\\Design0013\\e10_DOE.aedt',
 'E:\\KDH\\e10\\e10_DOE\\e10_DOE.opd\\AMOP\\Design0014\\e10_DOE.aedt',
 'E:\\

In [ ]:
"""
강건한 AEDT 파일 일괄 처리 루프

기능:
1. 현재 열린 프로젝트와 처리할 파일 비교
2. Parametric Setup 존재 여부 확인
3. CSV 결과 검증 후 필요한 경우에만 실행
4. 오류 발생 시 다음 파일로 자동 이동
"""

import os
from pathlib import Path
from aedt_csv_validator import validate_and_display_csv

# 처리 결과 수집
processing_results = []

# Parametric Sweep 설정
setup_name = "ParametricSetup1"
ipeak_steps = 5
phase_steps = 6


for file_idx, fileIndex in enumerate(file_paths[1:300], 1):
    file_path = Path(fileIndex)
    file_name = file_path.name
    
    print(f"\n{'='*80}")
    print(f"[{file_idx}/{len(file_paths[1:300])}] 📁 {file_name}")
    print(f"{'='*80}")
    
    try:
        # ===== 1. 프로젝트 로드 확인 =====
        current_project_path = m2d.project_path
        target_project_path = str(file_path.absolute())
        
        print(f"\n🔍 Step 1: 프로젝트 확인")
        print(f"  - 현재 열린 프로젝트 경로: {current_project_path}")
        print(f"  - 처리할 프로젝트 경로: {target_project_path}")
        
        # 경로 비교 (대소문자 무시, 정규화)
        current_normalized = Path(current_project_path).resolve()
        target_normalized = Path(target_project_path).resolve()
        
        is_same_project = current_normalized == target_normalized
        
        # 다른 프로젝트가 열려있으면 닫고 새로 열기
        if not is_same_project:
            print(f"  ℹ️  프로젝트 전환 필요")
            print(f"  📂 현재: ...{str(current_normalized)[-50:]}")
            print(f"  📂 목표: ...{str(target_normalized)[-50:]}")
            
            try:
                m2d.close_project()
                print(f"  ✅ 이전 프로젝트 닫기 완료")
            except Exception as e:
                print(f"  ⚠️ 프로젝트 닫기 실패 (무시): {e}")
            
            print(f"  📂 프로젝트 로드 중: {file_name}")
            m2d.load_project(fileIndex)
            print(f"  ✅ 프로젝트 로드 완료")
            
            # 로드 후 경로 재확인
            loaded_path = Path(m2d.project_path).resolve()
            if loaded_path != target_normalized:
                print(f"  ⚠️ 로드된 경로가 예상과 다름:")
                print(f"    예상: {target_normalized}")
                print(f"    실제: {loaded_path}")
        else:
            print(f"  ✅ 이미 올바른 프로젝트가 열려있음")
            print(f"  📂 경로: ...{str(current_normalized)[-60:]}")
        
        # ===== 2. Excitation 설정 =====
        print(f"\n⚡ Step 2: Excitation 설정")
        try:
            excitObj = m2d.excitation_objects
            
            # 필요한 excitation 확인
            required_excitations = ['WG_Ph1_P1', 'WG_Ph2_P1', 'WG_Ph3_P1']
            missing_excitations = [name for name in required_excitations if name not in excitObj]
            
            if missing_excitations:
                print(f"  ❌ 누락된 Excitation: {missing_excitations}")
                raise KeyError(f"필수 Excitation이 없습니다: {missing_excitations}")
            
            # Excitation 객체 가져오기
            Ph1Obj = excitObj['WG_Ph1_P1']
            Ph2Obj = excitObj['WG_Ph2_P1']
            Ph3Obj = excitObj['WG_Ph3_P1']
            
            # 전류 수식 설정
            ph1Current = 'IPeak  * sin(MachineRPM/1rpm*NumPoles/60*pi*time+PhaseAdvance-0deg+0)'
            ph2Current = 'IPeak  * sin(MachineRPM/1rpm*NumPoles/60*pi*time+PhaseAdvance-240deg+0)'
            ph3Current = 'IPeak  * sin(MachineRPM/1rpm*NumPoles/60*pi*time+PhaseAdvance-120deg+0)'
            
            # 전류 적용
            Ph1Obj.update_property(prop_name='Current', prop_value=ph1Current)
            Ph2Obj.update_property(prop_name='Current', prop_value=ph2Current)
            Ph3Obj.update_property(prop_name='Current', prop_value=ph3Current)
            
            print(f"  ✅ Excitation 설정 완료 (3-phase)")
            
        except Exception as e:
            print(f"  ❌ Excitation 설정 실패: {e}")
            processing_results.append({
                'file': file_name,
                'status': '❌ Excitation 설정 실패',
                'error': str(e)
            })
            continue
        
        # ===== 3. Parametric Setup 확인 =====
        print(f"\n🔧 Step 3: Parametric Setup 확인")
        param = m2d.parametrics
        oModule = param.optimodule
        
        # 기존 Optimetrics setup 목록 확인
        existing_setups = oModule.GetChildNames()
        setup_exists = setup_name in existing_setups
        
        print(f"  - 기존 Optimetrics Setup 목록: {existing_setups}")
        print(f"  - '{setup_name}' 존재 여부: {'✅ 있음' if setup_exists else '❌ 없음'}")
        
        # ===== 4. CSV 결과 검증 =====
        print(f"\n📊 Step 4: 기존 결과 확인")
        
        # CSV 파일 경로 생성
        current_aedt_path = m2d.project_path
        aedt_dir = Path(current_aedt_path).parent
        aedt_filename = Path(current_aedt_path).stem
        csv_filename = f"{aedt_filename}_{setup_name}_Result.csv"
        csv_path = aedt_dir / csv_filename
        
        # 결과 검증 (verbose=False로 간단하게)
        validation_result = validate_and_display_csv(
            m2d_obj=m2d,
            setup_name=setup_name,
            expected_ipeak_steps=ipeak_steps,
            expected_phase_steps=phase_steps,
            verbose=False
        )
        
        results_complete = validation_result['is_complete']
        csv_exists = validation_result['csv_exists']
        
        print(f"  - CSV 파일: {'✅ 존재' if csv_exists else '❌ 없음'}")
        if csv_exists:
            print(f"  - 결과 완성도: {validation_result['actual_count']}/{validation_result['expected_count']} "
                  f"({validation_result['completion_rate']:.1f}%)")
        
        # ===== 5. 처리 결정 로직 =====
        print(f"\n🎯 Step 5: 처리 결정")
        
        if results_complete:
            print(f"  ✅ 결과가 이미 완전함 - Sweep 생략")
            processing_results.append({
                'file': file_name,
                'status': '✅ Already Complete',
                'completion_rate': validation_result['completion_rate']
            })
            continue
        
        # 예상 Sweep 설정 정의
        expected_sweep_config = {
            "IPeak": {"Data": "LINC 10A 650.53A 5", "steps": ipeak_steps},
            "PhaseAdvance": {"Data": "LINC 0deg 90deg 6", "steps": phase_steps}
        }
        
        # Setup이 없으면 생성
        if not setup_exists:
            print(f"  📝 Parametric Setup 생성 중...")
            try:
                oModule.InsertSetup("OptiParametric", 
                    [
                        "NAME:ParametricSetup1",
                        "IsEnabled:=", True,
                        [
                            "NAME:ProdOptiSetupDataV2",
                            "SaveFields:=", True,
                            "CopyMesh:=", False,
                            "SolveWithCopiedMeshOnly:=", False
                        ],
                        "InterpolationPoints:=", 0,
                        ["NAME:StartingPoint"],
                        "Sim. Setups:=", ["Setup1"],
                        [
                            "NAME:Sweeps",
                            [
                                "NAME:SweepDefinition",
                                "Variable:=", "IPeak",
                                "Data:=", "LINC 10A 650.53A 5",
                                "OffsetF1:=", False,
                                "Synchronize:=", 0
                            ],
                            [
                                "NAME:SweepDefinition",
                                "Variable:=", "PhaseAdvance",
                                "Data:=", "LINC 0deg 90deg 6",
                                "OffsetF1:=", False,
                                "Synchronize:=", 0
                            ]
                        ],
                        ["NAME:Sweep Operations"],
                        ["NAME:Goals"]
                    ])
                print(f"  ✅ Parametric Setup 생성 완료")
            except Exception as e:
                print(f"  ❌ Parametric Setup 생성 실패: {e}")
                processing_results.append({
                    'file': file_name,
                    'status': '❌ Setup 생성 실패',
                    'error': str(e)
                })
                continue
        else:
            print(f"  ℹ️  Parametric Setup이 이미 존재함")
            
            # ===== 5-1. 기존 Setup 설정 검증 =====
            print(f"\n🔍 Step 5-1: Parametric Setup 설정 검증")
            try:
                cursimul = oModule.GetChildObject(setup_name)
                
                # Sweep 변수 확인
                sweep_vars = cursimul.GetSweepVariables()
                print(f"  - Sweep 변수: {sweep_vars}")
                
                # 설정 불일치 플래그
                config_mismatch = False
                
                # 예상 변수 확인
                expected_vars = list(expected_sweep_config.keys())
                if set(sweep_vars) != set(expected_vars):
                    print(f"  ⚠️ Sweep 변수 불일치:")
                    print(f"    예상: {expected_vars}")
                    print(f"    실제: {sweep_vars}")
                    config_mismatch = True
                else:
                    print(f"  ✅ Sweep 변수 일치: {sweep_vars}")
                    
                    # 각 변수의 범위 확인
                    for var in sweep_vars:
                        try:
                            var_data = cursimul.GetSweepData(var)
                            expected_data = expected_sweep_config[var]["Data"]
                            
                            # 간단한 문자열 비교 (공백 제거)
                            var_data_normalized = var_data.replace(" ", "").upper()
                            expected_data_normalized = expected_data.replace(" ", "").upper()
                            
                            if var_data_normalized != expected_data_normalized:
                                print(f"  ⚠️ {var} 범위 불일치:")
                                print(f"    예상: {expected_data}")
                                print(f"    실제: {var_data}")
                                config_mismatch = True
                            else:
                                print(f"  ✅ {var} 범위 일치: {var_data}")
                        except Exception as e:
                            print(f"  ⚠️ {var} 범위 확인 실패: {e}")
                            config_mismatch = True
                
                # 설정이 일치하지 않으면 Setup 삭제 후 재생성
                if config_mismatch:
                    print(f"\n  ⚠️ Setup 설정 불일치 - 재생성 필요")
                    print(f"  🗑️  기존 Setup 삭제 중...")
                    try:
                        oModule.DeleteSetups([setup_name])
                        print(f"  ✅ 기존 Setup 삭제 완료")
                        
                        print(f"  📝 Parametric Setup 재생성 중...")
                        oModule.InsertSetup("OptiParametric", 
                            [
                                "NAME:ParametricSetup1",
                                "IsEnabled:=", True,
                                [
                                    "NAME:ProdOptiSetupDataV2",
                                    "SaveFields:=", True,
                                    "CopyMesh:=", False,
                                    "SolveWithCopiedMeshOnly:=", False
                                ],
                                "InterpolationPoints:=", 0,
                                ["NAME:StartingPoint"],
                                "Sim. Setups:=", ["Setup1"],
                                [
                                    "NAME:Sweeps",
                                    [
                                        "NAME:SweepDefinition",
                                        "Variable:=", "IPeak",
                                        "Data:=", "LINC 10A 650.53A 5",
                                        "OffsetF1:=", False,
                                        "Synchronize:=", 0
                                    ],
                                    [
                                        "NAME:SweepDefinition",
                                        "Variable:=", "PhaseAdvance",
                                        "Data:=", "LINC 0deg 90deg 6",
                                        "OffsetF1:=", False,
                                        "Synchronize:=", 0
                                    ]
                                ],
                                ["NAME:Sweep Operations"],
                                ["NAME:Goals"]
                            ])
                        print(f"  ✅ Parametric Setup 재생성 완료")
                    except Exception as e:
                        print(f"  ❌ Setup 재생성 실패: {e}")
                        processing_results.append({
                            'file': file_name,
                            'status': '❌ Setup 재생성 실패',
                            'error': str(e)
                        })
                        continue
                else:
                    print(f"  ✅ Setup 설정 검증 통과")
                    
            except Exception as e:
                print(f"  ⚠️ Setup 설정 검증 실패: {e}")
                print(f"  ℹ️  기존 Setup 그대로 사용")
        
        # ===== 6. Parametric Sweep 실행 =====
        print(f"\n⚙️  Step 6: Parametric Sweep 실행 결정")
        
        # CSV가 불완전하면 Setup 삭제 후 재실행
        if csv_exists and not results_complete:
            print(f"  ⚠️ 기존 결과가 불완전함 ({validation_result['actual_count']}/{validation_result['expected_count']})")
            print(f"  🗑️  기존 Setup 삭제 후 재실행")
            
            try:
                # Setup 삭제
                oModule.DeleteSetups([setup_name])
                print(f"  ✅ 기존 Setup 삭제 완료")
                
                # Setup 재생성
                print(f"  📝 Parametric Setup 재생성 중...")
                oModule.InsertSetup("OptiParametric", 
                    [
                        "NAME:ParametricSetup1",
                        "IsEnabled:=", True,
                        [
                            "NAME:ProdOptiSetupDataV2",
                            "SaveFields:=", True,
                            "CopyMesh:=", False,
                            "SolveWithCopiedMeshOnly:=", False
                        ],
                        "InterpolationPoints:=", 0,
                        ["NAME:StartingPoint"],
                        "Sim. Setups:=", ["Setup1"],
                        [
                            "NAME:Sweeps",
                            [
                                "NAME:SweepDefinition",
                                "Variable:=", "IPeak",
                                "Data:=", "LINC 10A 650.53A 5",
                                "OffsetF1:=", False,
                                "Synchronize:=", 0
                            ],
                            [
                                "NAME:SweepDefinition",
                                "Variable:=", "PhaseAdvance",
                                "Data:=", "LINC 0deg 90deg 6",
                                "OffsetF1:=", False,
                                "Synchronize:=", 0
                            ]
                        ],
                        ["NAME:Sweep Operations"],
                        ["NAME:Goals"]
                    ])
                print(f"  ✅ Parametric Setup 재생성 완료")
                
                # CSV 파일도 삭제 (불완전한 결과 제거)
                if csv_path.exists():
                    csv_path.unlink()
                    print(f"  🗑️  불완전한 CSV 파일 삭제: {csv_filename}")
                
            except Exception as e:
                print(f"  ❌ Setup 재생성 실패: {e}")
                processing_results.append({
                    'file': file_name,
                    'status': '❌ Setup 재생성 실패 (불완전)',
                    'error': str(e)
                })
                continue
        
        # CSV가 없거나 재생성한 경우에만 실행
        if not csv_exists or (csv_exists and not results_complete):
            try:
                print(f"  🔄 Sweep 실행 중... (예상 시간: 수십 분 이상)")
                cursimul = oModule.GetChildObject(setup_name)
                cursimul.StartAnalyze()
                print(f"  ✅ Sweep 실행 완료")
                
                # ===== 7. CSV Export =====
                print(f"\n💾 Step 7: 결과 Export")
                try:
                    oModule.ExportOptimetricsResult(setup_name, str(csv_path), False)
                    print(f"  ✅ CSV Export 완료: {csv_filename}")
                    
                    # 최종 검증
                    final_validation = validate_and_display_csv(
                        m2d_obj=m2d,
                        setup_name=setup_name,
                        expected_ipeak_steps=ipeak_steps,
                        expected_phase_steps=phase_steps,
                        verbose=False
                    )
                    
                    processing_results.append({
                        'file': file_name,
                        'status': '✅ Sweep 완료' if final_validation['is_complete'] else '⚠️ Sweep 완료 (불완전)',
                        'completion_rate': final_validation['completion_rate']
                    })
                    
                except Exception as e:
                    print(f"  ⚠️ CSV Export 실패: {e}")
                    processing_results.append({
                        'file': file_name,
                        'status': '⚠️ Sweep 완료 (Export 실패)',
                        'error': str(e)
                    })
                    
            except Exception as e:
                print(f"  ❌ Sweep 실행 실패: {e}")
                processing_results.append({
                    'file': file_name,
                    'status': '❌ Sweep 실행 실패',
                    'error': str(e)
                })
                continue
        else:
            print(f"  ℹ️  기존 결과가 완전함 - Sweep 생략")
            processing_results.append({
                'file': file_name,
                'status': '✅ 기존 결과 사용',
                'completion_rate': validation_result['completion_rate']
            })
    
    except Exception as e:
        print(f"\n❌ 파일 처리 중 예외 발생: {e}")
        import traceback
        traceback.print_exc()
        
        processing_results.append({
            'file': file_name,
            'status': '❌ 처리 실패',
            'error': str(e)
        })
        continue

# ===== 최종 결과 요약 =====
print("\n" + "=" * 80)
print("📊 처리 결과 요약")
print("=" * 80)

if processing_results:
    import pandas as pd
    df_results = pd.DataFrame(processing_results)
    display(df_results)
    
    print(f"\n✅ 성공: {sum('✅' in r['status'] for r in processing_results)}개")
    print(f"⚠️ 경고: {sum('⚠️' in r['status'] for r in processing_results)}개")
    print(f"❌ 실패: {sum('❌' in r['status'] for r in processing_results)}개")
else:
    print("ℹ️  처리된 파일이 없습니다.")

print("=" * 80)


[1/299] 📁 e10_DOE.aedt

🔍 Step 1: 프로젝트 확인
  - 현재 열린 프로젝트: e10_DOE
  - 처리할 프로젝트: e10_DOE
  ✅ 이미 올바른 프로젝트가 열려있음

⚡ Step 2: Excitation 설정
  ✅ Excitation 설정 완료 (3-phase)

🔧 Step 3: Parametric Setup 확인
  - 기존 Optimetrics Setup 목록: ['ParametricSetup1']
  - 'ParametricSetup1' 존재 여부: ✅ 있음

📊 Step 4: 기존 결과 확인
  - CSV 파일: ✅ 존재
  - 결과 완성도: 30/30 (100.0%)

🎯 Step 5: 처리 결정
  ✅ 결과가 이미 완전함 - Sweep 생략

[2/299] 📁 e10_DOE.aedt

🔍 Step 1: 프로젝트 확인
  - 현재 열린 프로젝트: e10_DOE
  - 처리할 프로젝트: e10_DOE
  ✅ 이미 올바른 프로젝트가 열려있음

⚡ Step 2: Excitation 설정
  ✅ Excitation 설정 완료 (3-phase)

🔧 Step 3: Parametric Setup 확인
  - 기존 Optimetrics Setup 목록: ['ParametricSetup1']
  - 'ParametricSetup1' 존재 여부: ✅ 있음

📊 Step 4: 기존 결과 확인
  - CSV 파일: ✅ 존재
  - 결과 완성도: 30/30 (100.0%)

🎯 Step 5: 처리 결정
  ✅ 결과가 이미 완전함 - Sweep 생략

[2/299] 📁 e10_DOE.aedt

🔍 Step 1: 프로젝트 확인
  - 현재 열린 프로젝트: e10_DOE
  - 처리할 프로젝트: e10_DOE
  ✅ 이미 올바른 프로젝트가 열려있음

⚡ Step 2: Excitation 설정
  ✅ Excitation 설정 완료 (3-phase)

🔧 Step 3: Parametric Setup 확인
  - 기존 Optimetrics Set

: 

: 

In [ ]:
# Parametric Sweep 결과 검증 (validate_and_display_csv 함수 사용)
from aedt_csv_validator import validate_and_display_csv

# Parametric Sweep 설정
# IPeak: 5 steps (10A ~ 650.53A)
# PhaseAdvance: 6 steps (0deg ~ 90deg)
ipeak_steps = 5
phase_steps = 6

# 검증 실행 (Maxwell2d 객체로부터 자동으로 경로 추출)
validation_result = validate_and_display_csv(
    m2d_obj=m2d,
    setup_name="ParametricSetup1",
    expected_ipeak_steps=ipeak_steps,
    expected_phase_steps=phase_steps,
    verbose=True
)

In [ ]:
param_setups = m2d.parametrics.get_setup_names()


## FLD Export Function Definition

In [ ]:
m2dpost=m2d.post
all_objects = m2d.modeler.object_names

# Modelplotter=m2dpost.get_model_plotter_geometries(generate_mesh=True,get_objects_from_aedt=True)

### mesh export as *.case 

In [ ]:
import os 
desktop = ansys.aedt.core.Desktop() 
pjtPath=desktop.project_path()
prjName=desktop.active_project().GetName()
filePath=os.path.join(pjtPath, prjName) 
pjt=desktop.load_project(filePath)
setup=pjt.get_setup(name='Setup1')
FieldReporter=pjt.get_module("FieldsReporter")


In [ ]:
SetupObj=m2d.get_setup('Setup1')
{"Time":SetupObj.props['MaxTimeStep']}
import ansys.aedt.core.visualization.plot.pyvista as AEDTvista
AEDTvista
import ansys.aedt.core.visualization.post as AEDTpost
AEDTpost

## plot Field

In [ ]:
m2dpost.get_solution_data
solutions = m2d.post.get_solution_data(
    primary_sweep_variable="Time", domain="Sweep"
)

timeSteps=solutions.variation_values(variation='Time')
timeUnit=solutions.units_sweeps['Time']

In [ ]:
str(timeSteps[0])+timeUnit

In [ ]:
import os
from pathlib import Path

# Export 경로 설정
export_dir = r"E:\KDH\e10\e10_DOE\aedtplt_exports"
Path(export_dir).mkdir(parents=True, exist_ok=True)

oDesign = m2d.odesign
oModule = oDesign.GetModule("FieldsReporter")


In [ ]:
time_str = f"Time='{time_value}{timeUnit}'"
time_value
timeUnit

In [ ]:
file_name = f"A_Vector_Time_{idx:03d}.aedtplt"
file_name

In [ ]:
file_path = os.path.join(export_dir, file_name)
file_path

In [ ]:
# Solution context 설정
oModule.SetPlotsViewSolutionContext(
    ["A_Vector1"], 
    "Setup1 : Transient", 
    time_str
)

In [ ]:
oModule.ExportFieldPlot("A_Vector1", False, file_path)


In [ ]:
for idx, time_value in enumerate(timeSteps):
    try:
        # Time 문자열 생성 (예: "Time='0.00011494252873563217s'")
        time_str = f"Time='{time_value}{timeUnit}'"
        
        # 파일명 생성 (예: A_Vector_Time_000.aedtplt)
        file_name = f"A_Vector_Time_{idx:03d}.aedtplt"
        file_path = os.path.join(export_dir, file_name)
        
        # Solution context 설정
        oModule.SetPlotsViewSolutionContext(
            ["A_Vector1"], 
            "Setup1 : Transient", 
            time_str
        )
        
        # Field plot export
        oModule.ExportFieldPlot("A_Vector1", False, file_path)
        
        exported_files.append({
            'index': idx,
            'time_value': time_value,
            'time_unit': timeUnit,
            'file_path': file_path,
            'file_name': file_name
        })
        
        print(f"  ✅ [{idx+1}/{len(timeSteps)}] {file_name} - Time: {time_value}{timeUnit}")
        
    except Exception as e:
        print(f"  ❌ [{idx+1}/{len(timeSteps)}] 실패 - Time: {time_value}{timeUnit}")
        print(f"     오류: {e}")

print("\n" + "=" * 70)
print(f"📊 Export 완료!")
print(f"✅ 성공: {len(exported_files)}개")
print(f"❌ 실패: {len(timeSteps) - len(exported_files)}개")
print("=" * 70)

# 결과를 DataFrame으로 변환
if exported_files:
    df_exported = pd.DataFrame(exported_files)
    print("\n📋 Export된 파일 목록:")
    display(df_exported[['index', 'time_value', 'file_name']])

In [ ]:
m2dpost

In [ ]:
m2dpost.ani

In [ ]:
oDesign.ChangeProperty(
	[
		"NAME:AllTabs",
		[
			"NAME:LocalVariableTab",
			[
				"NAME:PropServers", 
				"LocalVariables"
			],
			[
				"NAME:ChangedProps",
				[
					"NAME:IPeak",
					"Value:="		, "10A"
				],
				[
					"NAME:PhaseAdvance",
					"Value:="		, "18deg"
				]
			]
		]
	])

In [ ]:
for time in timeSteps:
    plot=m2dpost.plot_field(
        quantity="",
        assignment=all_objects,
        plot_type="Surface",
        show=False,
        mesh_on_fields=True,
        file_format="case",
        plot_cad_objs=True,
        intrinsics={"Time":str(time)+timeUnit}
    )

In [ ]:
m2dpost.field

In [ ]:
plot=m2dpost.plot_field(
    quantity="A_vector",
    assignment=all_objects,
    plot_type="Surface",
    show=False,
    mesh_on_fields=True,
    file_format="aedtplt",
    intrinsics={"Time":"All"}
)

In [ ]:
import pyvista as pv

# 인터랙티브 렌더링 설정
pv.set_jupyter_backend('trame')  # 'panel'이 가장 인터랙티브함

# case_file = r"F:\KDH\Thesis\JEET\e10_tuto\e10_tutorial_ANSYSEM_2D.pyaedt\Motor-CAD_e10_tutorial\Mag_B_KXZ0HU.case"
case_file=r"E:\KDH\e10\e10_DOE\e10_DOE.aedtresults\Motor-CAD e10_tutorial.results\Mag_B_KXZ0HU.case"
reader = pv.get_reader(case_file)
mesh = reader.read()

print(f"📦 데이터 타입: {type(mesh).__name__}")

if isinstance(mesh, pv.MultiBlock):
    print(f"🔹 MultiBlock: {mesh.n_blocks}개 블록")
    
    # 인터랙티브 Plotter 생성
    plotter = pv.Plotter(notebook=True)
    
    # 각 블록 추가
    for i, block in enumerate(mesh):
        if block is not None and block.n_points > 0:
            # 첫 번째 유효한 필드 찾기
            if block.point_data:
                field_name = list(block.point_data.keys())[0]
                print(f"블록 {i}: {field_name} 필드 사용")
                
                # 필드 컬러맵으로 표시
                plotter.add_mesh(
                    block,
                    scalars=field_name,
                    show_edges=True,  # ✨ Mesh 표시
                    edge_color='black',  # Mesh 선 색상
                    line_width=0.5,  # Mesh 선 두께
                    cmap='jet',
                    opacity=1.0,
                    scalar_bar_args={
                        'title': field_name,
                        'vertical': True,
                        'height': 0.25,
                        'width': 0.05,
                        'position_x': 0.85,
                        'position_y': 0.05
                    }
                )
    
    # 카메라 및 조명 설정
    plotter.enable_anti_aliasing()
    plotter.add_axes()
    
    # 인터랙티브 Plot 표시
    plotter.show(jupyter_backend='trame')
    
else:
    # 단일 메시
    print(f"📊 Points: {mesh.n_points}")
    print(f"📐 Cells: {mesh.n_cells}")
    
    if mesh.point_data:
        field_name = list(mesh.point_data.keys())[0]
        
        plotter = pv.Plotter(notebook=True)
        plotter.add_mesh(
            mesh,
            scalars=field_name,
            show_edges=True,  # ✨ Mesh 표시
            edge_color='black',
            line_width=0.5,
            cmap='jet',
            scalar_bar_args={'title': field_name}
        )
        plotter.enable_anti_aliasing()
        plotter.add_axes()
        plotter.show(jupyter_backend='trame')

In [ ]:

gif = m2d.post.plot_animated_field(
    quantity="Mag_B",
    assignment=all_objects,
    plot_type="Surface",
    intrinsics={"Time": "0s"},
    variation_variable="Time",
    variations=timesteps,
    show=False,
    export_gif=False,
)
gif.isometric_view = False
gif.camera_position = [15, 15, 80]
gif.focal_point = [15, 15, 0]
gif.roll_angle = 0
gif.elevation_angle = 0
gif.azimuth_angle = 0

# Set off_screen to False to visualize the animation.
# gif.off_screen = False
gif.animate()